# Exercício 5 — Pilha e fila com invariantes e árvore de expressão

**Henrique Goldstein Maluhy Mendes Amaral** · DR1_AT — Algoritmos e Estruturas de Dados · Infnet

Para rodar: **Ambiente de execução → Executar tudo**. O notebook é independente, não precisa de nenhum arquivo externo.

**Roteiro:** (1) `Stack` e `Queue` em vetor de capacidade fixa, com exceções próprias · (2) teste longo das invariantes, incluindo o wraparound da fila · (3) árvore de expressão a partir da pós-fixa, com detecção de entradas inválidas · (4) avaliação recursiva × iterativa e o risco da profundidade · (5) casos de borda

In [1]:
import random
import sys
import time

import pandas as pd

random.seed(42)

## 1. `Stack` e `Queue` com capacidade fixa

As duas usam um **vetor estático** (`[None] * capacidade`), criado uma vez só — como na aula do simulador de fila de impressão.

- **Pilha:** `topo` guarda o índice do último elemento; `-1` quer dizer vazia.
- **Fila circular:** `inicio` aponta para o primeiro, `fim` para a próxima posição livre, e `total` diz quantos há. Quando `fim` chega no final do vetor, ele **volta para a posição 0** (`% capacidade`) — é o wraparound. Assim a fila reaproveita o espaço liberado na frente, sem nunca deslocar elementos.

Cada situação de erro tem a **sua própria exceção**, com uma mensagem dizendo o que aconteceu.

In [2]:
class StackOverflowError(Exception):
    """push em pilha cheia."""

class StackUnderflowError(Exception):
    """pop ou peek em pilha vazia."""

class QueueOverflowError(Exception):
    """enqueue em fila cheia."""

class QueueUnderflowError(Exception):
    """dequeue ou peek em fila vazia."""


class Stack:
    def __init__(self, capacidade):
        if capacidade < 1:
            raise ValueError(f"a capacidade precisa ser pelo menos 1, recebi {capacidade}")
        self.capacidade = capacidade
        self.itens = [None] * capacidade
        self.topo = -1

    def is_empty(self):
        return self.topo == -1

    def is_full(self):
        return self.topo == self.capacidade - 1

    def __len__(self):
        return self.topo + 1

    def push(self, item):
        if self.is_full():
            raise StackOverflowError(
                f"push({item!r}) em pilha cheia (capacidade {self.capacidade})")
        self.topo += 1
        self.itens[self.topo] = item

    def pop(self):
        if self.is_empty():
            raise StackUnderflowError("pop em pilha vazia")
        item = self.itens[self.topo]
        self.itens[self.topo] = None     # limpo a posição para não segurar a referência
        self.topo -= 1
        return item

    def peek(self):
        if self.is_empty():
            raise StackUnderflowError("peek em pilha vazia")
        return self.itens[self.topo]


class Queue:
    def __init__(self, capacidade):
        if capacidade < 1:
            raise ValueError(f"a capacidade precisa ser pelo menos 1, recebi {capacidade}")
        self.capacidade = capacidade
        self.itens = [None] * capacidade
        self.inicio = 0
        self.fim = 0
        self.total = 0
        self.voltas = 0          # quantas vezes o `fim` deu a volta no vetor (só para medir)

    def is_empty(self):
        return self.total == 0

    def is_full(self):
        return self.total == self.capacidade

    def __len__(self):
        return self.total

    def enqueue(self, item):
        if self.is_full():
            raise QueueOverflowError(
                f"enqueue({item!r}) em fila cheia (capacidade {self.capacidade})")
        self.itens[self.fim] = item
        self.fim = (self.fim + 1) % self.capacidade   # wraparound
        if self.fim == 0:
            self.voltas += 1
        self.total += 1

    def dequeue(self):
        if self.is_empty():
            raise QueueUnderflowError("dequeue em fila vazia")
        item = self.itens[self.inicio]
        self.itens[self.inicio] = None
        self.inicio = (self.inicio + 1) % self.capacidade
        self.total -= 1
        return item

    def peek(self):
        if self.is_empty():
            raise QueueUnderflowError("peek em fila vazia")
        return self.itens[self.inicio]

    def em_ordem(self):
        """Os elementos do primeiro ao último, seguindo o buffer circular."""
        return [self.itens[(self.inicio + i) % self.capacidade] for i in range(self.total)]

### Invariantes

Depois de **qualquer** operação, as duas estruturas precisam satisfazer:

- **pilha:** `-1 <= topo < capacidade`; as posições acima do topo estão vazias (`None`);
- **fila:** `0 <= total <= capacidade`; `fim == (inicio + total) % capacidade`; as posições fora da janela ocupada estão vazias.

In [3]:
def verificar_pilha(p):
    assert -1 <= p.topo < p.capacidade
    assert all(x is None for x in p.itens[p.topo + 1:]), "sobrou lixo acima do topo"


def verificar_fila(f):
    assert 0 <= f.total <= f.capacidade
    assert f.fim == (f.inicio + f.total) % f.capacidade, "inicio, fim e total não batem"
    ocupadas = {(f.inicio + i) % f.capacidade for i in range(f.total)}
    for i in range(f.capacidade):
        if i not in ocupadas:
            assert f.itens[i] is None, f"posição {i} deveria estar vazia"

## 2. Teste longo: 10.000 operações sorteadas

Uma pilha e uma fila de **capacidade 8** recebem 10.000 operações aleatórias. Em paralelo, uma lista do Python faz o papel de **gabarito** (só no teste). A cada operação eu confiro:

- se a estrutura estava cheia/vazia, a exceção certa **tem** que acontecer;
- senão, o resultado tem que ser igual ao do gabarito;
- as invariantes continuam valendo.

In [4]:
pilha, fila = Stack(8), Queue(8)
gab_pilha, gab_fila = [], []
contagem = {"overflow pilha": 0, "underflow pilha": 0, "overflow fila": 0, "underflow fila": 0}

for passo in range(10_000):
    # pilha
    if random.random() < 0.5:
        try:
            pilha.push(passo)
            assert len(gab_pilha) < 8
            gab_pilha.append(passo)
        except StackOverflowError:
            assert len(gab_pilha) == 8          # só pode estourar se estiver mesmo cheia
            contagem["overflow pilha"] += 1
    else:
        try:
            assert pilha.pop() == gab_pilha.pop()
        except StackUnderflowError:
            assert not gab_pilha
            contagem["underflow pilha"] += 1
        except IndexError:                      # o gabarito estava vazio e a pilha não reclamou
            raise AssertionError("a pilha devia ter levantado StackUnderflowError")
    verificar_pilha(pilha)

    # fila
    if random.random() < 0.5:
        try:
            fila.enqueue(passo)
            assert len(gab_fila) < 8
            gab_fila.append(passo)
        except QueueOverflowError:
            assert len(gab_fila) == 8
            contagem["overflow fila"] += 1
    else:
        try:
            assert fila.dequeue() == gab_fila.pop(0)   # FIFO: sai o mais antigo
        except QueueUnderflowError:
            assert not gab_fila
            contagem["underflow fila"] += 1
        except IndexError:
            raise AssertionError("a fila devia ter levantado QueueUnderflowError")
    verificar_fila(fila)
    assert fila.em_ordem() == gab_fila

print("10.000 operações: resultados iguais ao gabarito e invariantes válidas em todos os passos")
print(f"a fila deu {fila.voltas} voltas no vetor de 8 posições (wraparound) sem perder a ordem FIFO\n")
pd.DataFrame([contagem])

10.000 operações: resultados iguais ao gabarito e invariantes válidas em todos os passos
a fila deu 552 voltas no vetor de 8 posições (wraparound) sem perder a ordem FIFO



,overflow pilha,underflow pilha,overflow fila,underflow fila
0,497,567,548,625


As quatro exceções aconteceram centenas de vezes cada, **sempre** no momento certo — nunca antes (estrutura com espaço) nem depois (estrutura já cheia ou vazia). E a fila deu centenas de voltas no vetor de 8 posições sem perder a ordem.

**Custo:** `push`, `pop`, `enqueue`, `dequeue` e `peek` são todos **O(1)**: mexem num índice só. A fila circular é o que garante isso — uma fila que tirasse sempre da posição 0 de uma lista (`pop(0)`) teria que deslocar todo mundo, O(n) por `dequeue`.

## 3. Árvore de expressão a partir da pós-fixa

Na pós-fixa o operador vem **depois** dos operandos (`3 4 +` quer dizer `3 + 4`). Montar a árvore com uma pilha:

1. **operando** → vira um nó folha e vai para a pilha;
2. **operador** → tira dois nós da pilha (o primeiro que sai é o da **direita**), cria um nó com eles como filhos e empilha o resultado;
3. no fim, **tem que sobrar exatamente um nó**: a raiz.

Duas entradas inválidas são detectadas, dizendo **em que posição** o problema apareceu:
- operador sem dois operandos na pilha (`3 +`);
- operandos sobrando no fim (`3 4 5 +`).

In [5]:
OPERADORES = {"+", "-", "*", "/", "^"}


class ExpressaoInvalidaError(Exception):
    """A string não é uma expressão pós-fixa válida."""


class NoExpressao:
    def __init__(self, valor, esquerda=None, direita=None):
        self.valor = valor
        self.esquerda = esquerda
        self.direita = direita

    def eh_folha(self):
        return self.esquerda is None and self.direita is None


def construir_arvore(posfixa):
    tokens = posfixa.split()
    if not tokens:
        raise ExpressaoInvalidaError("expressão vazia")

    # a pilha nunca precisa guardar mais nós do que há tokens
    pilha = Stack(len(tokens))
    for posicao, token in enumerate(tokens, start=1):
        if token in OPERADORES:
            if len(pilha) < 2:
                raise ExpressaoInvalidaError(
                    f"o operador '{token}' (token {posicao}) precisa de dois operandos, "
                    f"mas a pilha tem {len(pilha)}")
            direita = pilha.pop()
            esquerda = pilha.pop()
            pilha.push(NoExpressao(token, esquerda, direita))
        else:
            try:
                numero = float(token)
            except ValueError:
                raise ExpressaoInvalidaError(
                    f"'{token}' (token {posicao}) não é número nem operador") from None
            pilha.push(NoExpressao(numero))

    if len(pilha) != 1:
        raise ExpressaoInvalidaError(
            f"sobraram {len(pilha)} operandos sem operador no fim da expressão")
    return pilha.pop()


def para_infixa(no):
    """Só para visualizar a árvore: reescreve com parênteses."""
    if no.eh_folha():
        return f"{no.valor:g}"
    return f"({para_infixa(no.esquerda)} {no.valor} {para_infixa(no.direita)})"


raiz = construir_arvore("3 4 + 2 *")
print("3 4 + 2 *  →", para_infixa(raiz))
raiz = construir_arvore("5 1 2 + 4 * + 3 -")
print("5 1 2 + 4 * + 3 -  →", para_infixa(raiz))

3 4 + 2 *  → ((3 + 4) * 2)
5 1 2 + 4 * + 3 -  → ((5 + ((1 + 2) * 4)) - 3)


## 4. Avaliando de duas formas

- **Recursiva (pós-ordem):** avalia a subárvore esquerda, depois a direita, e só então aplica o operador do nó. Fica curtinha, mas cada nível da árvore vira uma chamada empilhada na pilha do **Python**.
- **Iterativa com `Stack`:** faz a mesma pós-ordem com uma pilha **minha**. Cada nó entra primeiro como "ainda não avaliei os filhos" e depois como "filhos prontos, posso aplicar o operador".

As duas contam quantos nós visitaram, para comparar o trabalho.

In [6]:
def aplicar(operador, a, b):
    if operador == "+":
        return a + b
    if operador == "-":
        return a - b
    if operador == "*":
        return a * b
    if operador == "/":
        if b == 0:
            raise ZeroDivisionError("divisão por zero ao avaliar a expressão")
        return a / b
    return a ** b


def avaliar_recursivo(no, visitas):
    visitas[0] += 1
    if no.eh_folha():
        return no.valor
    a = avaliar_recursivo(no.esquerda, visitas)
    b = avaliar_recursivo(no.direita, visitas)
    return aplicar(no.valor, a, b)


def contar_nos(raiz):
    total, pilha = 0, [raiz]
    while pilha:
        no = pilha.pop()
        total += 1
        pilha.extend(f for f in (no.esquerda, no.direita) if f is not None)
    return total


def avaliar_iterativo(raiz, visitas):
    n = contar_nos(raiz)
    tarefas = Stack(2 * n)      # cada nó entra no máximo duas vezes
    valores = Stack(n)
    tarefas.push((raiz, False))
    while not tarefas.is_empty():
        no, filhos_prontos = tarefas.pop()
        if no.eh_folha():
            visitas[0] += 1
            valores.push(no.valor)
        elif filhos_prontos:
            b = valores.pop()
            a = valores.pop()
            valores.push(aplicar(no.valor, a, b))
        else:
            visitas[0] += 1
            tarefas.push((no, True))           # volto nele depois dos filhos
            tarefas.push((no.direita, False))
            tarefas.push((no.esquerda, False)) # empilhado por último = avaliado primeiro
    return valores.pop()


casos = {
    "3 4 + 2 *": 14,
    "5 1 2 + 4 * + 3 -": 14,
    "2 3 ^ 1 -": 7,
    "10 4 / 2 *": 5,
    "7": 7,
}
for expressao, esperado in casos.items():
    arvore = construir_arvore(expressao)
    v_rec, v_it = [0], [0]
    r = avaliar_recursivo(arvore, v_rec)
    i = avaliar_iterativo(arvore, v_it)
    assert r == i == esperado, expressao
    assert v_rec[0] == v_it[0] == contar_nos(arvore)   # as duas visitam cada nó uma vez
    print(f"{expressao:<22} = {r:g}   ({v_rec[0]} nó(s) visitado(s) nas duas versões)")

3 4 + 2 *              = 14   (5 nó(s) visitado(s) nas duas versões)
5 1 2 + 4 * + 3 -      = 14   (9 nó(s) visitado(s) nas duas versões)
2 3 ^ 1 -              = 7   (5 nó(s) visitado(s) nas duas versões)
10 4 / 2 *             = 5   (5 nó(s) visitado(s) nas duas versões)
7                      = 7   (1 nó(s) visitado(s) nas duas versões)


### O risco da recursão: profundidade

As duas versões fazem o **mesmo trabalho** (visitam cada nó uma vez, O(n)). A diferença é **onde fica a pilha**:

- na recursiva, cada nível da árvore é uma chamada guardada na pilha do Python, que tem limite;
- na iterativa, a pilha é minha e cresce na memória comum.

Uma expressão como `1 1 + 1 + 1 + ...` gera uma árvore que só cresce para a esquerda: a profundidade é igual ao número de operadores. Monto uma com profundidade **três vezes o limite de recursão** do Python:

In [7]:
limite = sys.getrecursionlimit()
profundidade = 3 * limite
expressao = "1 " + "1 + " * profundidade
arvore = construir_arvore(expressao)
print(f"limite de recursão do Python: {limite}")
print(f"expressão com {profundidade} operadores → árvore de {contar_nos(arvore)} nós\n")

try:
    avaliar_recursivo(arvore, [0])
    print("recursiva: terminou (o limite deste ambiente é maior que o esperado)")
except RecursionError:
    print(f"recursiva: RecursionError — estourou a pilha do Python")

visitas = [0]
inicio = time.perf_counter()
resultado = avaliar_iterativo(arvore, visitas)
duracao = time.perf_counter() - inicio
print(f"iterativa: resultado {resultado:g} (o esperado é {profundidade + 1}), "
      f"{visitas[0]} nós visitados em {duracao * 1000:.1f} ms")
assert resultado == profundidade + 1

limite de recursão do Python: 1000
expressão com 3000 operadores → árvore de 6001 nós

recursiva: RecursionError — estourou a pilha do Python
iterativa: resultado 3001 (o esperado é 3001), 6001 nós visitados em 8.3 ms


**Complexidade e riscos:**

| | tempo | memória extra | risco |
|---|---|---|---|
| montar a árvore | O(n) — um push/pop por token | O(n) nós | — |
| avaliação recursiva | O(n) | O(h) chamadas na pilha do Python | **quebra com `RecursionError`** se `h` passar do limite (~1.000) |
| avaliação iterativa | O(n) | O(h) na minha pilha | só depende da memória disponível |

`h` é a altura da árvore: em expressões balanceadas é perto de log₂ n, mas numa expressão encadeada como a de cima chega a `n/2`. A recursiva é mais fácil de ler e serve bem para expressões de tamanho normal; a iterativa é a que **não quebra** em entradas grandes ou maliciosas.

## 5. Casos de borda

In [8]:
invalidas = {
    "": "expressão vazia",
    "3 +": "operador sem dois operandos",
    "+": "operador sozinho",
    "3 4 5 +": "operandos sobrando",
    "3 4 x": "token desconhecido",
    "1 2 + +": "segundo operador sem operandos",
}
for expressao, motivo in invalidas.items():
    try:
        construir_arvore(expressao)
        raise AssertionError(f"'{expressao}' deveria ser inválida")
    except ExpressaoInvalidaError as erro:
        print(f"{motivo:<32} '{expressao}' → {erro}")

# divisão por zero é erro de avaliação, não de montagem
arvore = construir_arvore("1 0 /")
try:
    avaliar_iterativo(arvore, [0])
except ZeroDivisionError as erro:
    print(f"\n'1 0 /' monta a árvore, mas ao avaliar → {erro}")

# números negativos e com vírgula decimal
assert avaliar_recursivo(construir_arvore("-2 3.5 *"), [0]) == -7.0

# pilha e fila com capacidade 1
p = Stack(1)
p.push("a")
try:
    p.push("b")
except StackOverflowError as erro:
    print(f"\npilha de capacidade 1 → {erro}")
f = Queue(1)
f.enqueue("a")
assert f.dequeue() == "a"
try:
    f.dequeue()
except QueueUnderflowError as erro:
    print(f"fila de capacidade 1 → {erro}")

# capacidade inválida
try:
    Stack(0)
except ValueError as erro:
    print(f"Stack(0) → {erro}")

print("\ntodos os casos de borda passaram")

expressão vazia                  '' → expressão vazia
operador sem dois operandos      '3 +' → o operador '+' (token 2) precisa de dois operandos, mas a pilha tem 1
operador sozinho                 '+' → o operador '+' (token 1) precisa de dois operandos, mas a pilha tem 0
operandos sobrando               '3 4 5 +' → sobraram 2 operandos sem operador no fim da expressão
token desconhecido               '3 4 x' → 'x' (token 3) não é número nem operador
segundo operador sem operandos   '1 2 + +' → o operador '+' (token 4) precisa de dois operandos, mas a pilha tem 1

'1 0 /' monta a árvore, mas ao avaliar → divisão por zero ao avaliar a expressão

pilha de capacidade 1 → push('b') em pilha cheia (capacidade 1)
fila de capacidade 1 → dequeue em fila vazia
Stack(0) → a capacidade precisa ser pelo menos 1, recebi 0

todos os casos de borda passaram
